# From Anomaly Scores To Investigation

An anomaly score tells us how unusual an observation is under a particular reference and method. A flag selects it for attention. Investigation supplies evidence about what the departure means.

Today we connect those steps: choose a reference, decide which records to review, and evaluate what the selections accomplish. Wine chemistry supplies the observations; sensory ratings give us external evidence to interpret later.

## Continue Scores, Flags, And Investigation

Recall the two comparisons from the previous meeting: LOF compares local density with neighboring densities; Isolation Forest measures ease of isolation. Their scores can prioritize different observations.

## References

- [UCI Wine Quality](https://archive.ics.uci.edu/dataset/186/wine+quality): observations, sensory ratings, and CC BY 4.0 data attribution to Cortez and colleagues. Original files are included in `data/`.
- [Chandola, Banerjee, and Kumar, §2.2](https://arindam.cs.illinois.edu/papers/09/anomaly.pdf): point, contextual, and collective anomalies.
- [Scikit-learn anomaly detection](https://scikit-learn.org/stable/modules/outlier_detection.html): method interfaces, thresholds, and novelty detection.
- [Cortez et al., Table 1](https://projects.csail.mit.edu/wiki/pub/Evodesign/SensoryEvaluationsDatabase/winequality09.pdf): measurement units. Total sulfur dioxide is measured in mg/L.
- [SciPy hypergeometric distribution](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.hypergeom.html): exact check of the random-selection experiment.

## The Dataset

The [UCI Wine Quality dataset](https://archive.ics.uci.edu/dataset/186/wine+quality) contains 1,599 red-wine and 4,898 white-wine records from Portuguese vinho verde samples. Each row combines 11 chemical measurements with a sensory quality rating, the median of at least three expert assessments.

We will use the chemical measurements to identify unusual profiles. Later, the quality ratings will help us interpret what the detectors select. Load both files and inspect a few red-wine records.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import hypergeom
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler

# Run from this notebook's folder or the repository root.
data_dir = Path("data")
if not (data_dir / "winequality-red.csv").exists():
    data_dir = Path("content/lectures/10a-anomaly-decisions-and-validation/data")
red = pd.read_csv(data_dir / "winequality-red.csv", sep=";")
white = pd.read_csv(data_dir / "winequality-white.csv", sep=";")
red.index = pd.RangeIndex(1, len(red) + 1, name="record")
assert red.shape == (1599, 12) and not red.isna().any().any()
display(red.head())

Each row is one recorded sample. Residual sugar is in g/L, total sulfur dioxide in mg/L, and alcohol in percent by volume. The other chemical variables describe acidity, salts, density, and pH.

The files contain 240 exact repeated red-wine rows. We retain them in this analysis; their repetition contributes to the fitted reference.

## Anomalies And Their Reference

**An anomaly is an observation or pattern that departs from a specified reference.** Our initial question is: which wines have unusual chemical profiles? We will fit the detectors using chemistry alone.

**The reference is the population or fitted pattern used to assess unusualness.** Choosing it determines which comparisons the scores express.

### Point, Contextual, And Collective Anomalies

| Type | Definition | Example |
| --- | --- | --- |
| Point | An individual observation departs from its reference. | One unusual chemical profile among red wines. |
| Contextual | An observation is unusual under relevant conditions. | A sulfur dioxide measurement is unusual for red wine but common for white wine. |
| Collective | Related observations form an unexpected joint pattern. | Adjacent sensors show elevated readings when elevations ordinarily occur at scattered locations. |

The distinction affects the information needed. Chemical profiles support individual scoring; wine type supplies context; a spatial pattern requires sensor relationships.

The following code constructs two sensor panels containing the same readings. Compare their spatial arrangements.

In [ ]:
from matplotlib.colors import ListedColormap

scattered = np.ones((6, 6), dtype=int)
scattered[[0, 0, 1, 2, 3, 4, 5, 5], [0, 3, 5, 1, 4, 0, 2, 5]] = 2
patch = np.ones((6, 6), dtype=int)
patch[1:3, 1:5] = 2
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, values, title in zip(axes, [scattered, patch],
                            ["Scattered elevations", "Connected elevated region"], strict=True):
    ax.imshow(values, cmap=ListedColormap(["#edf1f4", "#b85a36"]), vmin=1, vmax=2)
    for row in range(6):
        for col in range(6):
            ax.text(col, row, str(values[row, col]), ha="center", va="center")
    ax.set(title=title, xlabel="Sensor column", ylabel="Sensor row")
fig.tight_layout()
plt.show()

Both panels contain eight readings of 2 and twenty-eight readings of 1. Suppose both values are familiar individually, but adjacent elevations are unexpected. The right-hand pattern then warrants investigation. Location supplies the information needed to distinguish the panels.

### Choose The Comparison Population

Compare the distribution of total sulfur dioxide within each wine type. The following plot uses a shared measurement axis and normalizes each histogram separately so the larger white-wine file does not dominate by count.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for frame, label, color in [(red, "Red wine", "#9b3346"),
                            (white, "White wine", "#277c8e")]:
    ax.hist(frame["total sulfur dioxide"], bins=np.arange(0, 451, 15),
            density=True, histtype="step", linewidth=2, label=label, color=color)
ax.axvline(150, color="black", linestyle="--", label="150 mg/L")
ax.set(xlabel="Total sulfur dioxide (mg/L)", ylabel="Density",
       title="The reference changes how unusual a measurement appears")
ax.legend()
plt.show()

A value of 150 mg/L exceeds about 99.4% of red-wine measurements but only 63.1% of white-wine measurements. What comparison would pooling the files obscure?

##### Answer

Pooling would obscure departures within wine type. The same measurement can be exceptional among red wines and ordinary among white wines. Use red wines as the reference for the main analysis.

## Score The Red-Wine Records

We will standardize the 11 chemical measurements and fit both detectors to the same red-wine records. Quality is excluded from the inputs.

Subtracting each variable's mean and dividing by its standard deviation makes the distance calculation comparable across measurement scales. Use 20 neighbors for a local comparison and 300 isolation trees to average random partitions. The fixed seed makes this starting comparison reproducible.

In [ ]:
chemistry = red.drop(columns="quality")
scaler = StandardScaler()
X = scaler.fit_transform(chemistry)

lof = LocalOutlierFactor(n_neighbors=20).fit(X)
forest = IsolationForest(
    n_estimators=300, max_samples=256, random_state=7130
).fit(X)

# Orient both scores so that larger means more unusual.
scores = pd.DataFrame({
    "LOF": -lof.negative_outlier_factor_,
    "Isolation Forest": -forest.score_samples(X),
}, index=red.index)
display(scores.head().round(3))

LOF compares local density; Isolation Forest aggregates isolation paths. The minus signs reverse the library's score orientation so that larger values mean more unusual. Each method has its own score scale, so compare rankings or decisions across methods.

## Set A Review Threshold

**A threshold is the score boundary used to assign flags.** With our orientation, flag records at or above it. **A review budget is the number of observations we can investigate.**

Use an illustrative budget of 50. The following cell obtains each method's fiftieth-highest score, then applies that full-precision threshold. The display rounds only for reading.

In [ ]:
budget = 50
thresholds = scores.apply(lambda column: column.nlargest(budget).iloc[-1])
flags = scores.ge(thresholds)
threshold_table = pd.DataFrame({
    "Threshold (rounded)": thresholds,
    "Records flagged": flags.sum(),
})
display(threshold_table.round(6))
print("Records selected by both:", int(flags.all(axis=1).sum()))

Each method selects 50 records here, with 19 in common. Their thresholds are approximately 1.506 and .554. The differing lists mean the detector choice changes whom we would investigate.

A fixed score threshold allows workload to vary. A fixed review budget lets the numerical cutoff change with the scores. Tied scores can make a threshold select more records than the requested budget; keep the rule explicit.

Moving a threshold changes the flags while leaving the fitted model and scores fixed. The consequences depend on which records enter the list and what investigation reveals about them.

## Compare Flags With External Evidence

The dataset includes a "quality" measure. This external outcome provides evidence separate from the anomaly score. This rating lets us ask whether low-rated wines occur more often among flagged chemical profiles.

**A proxy is an available measure used in place of the condition we want to assess.** Low ratings are a candidate proxy for a problematic wine, with an uncertain relationship to chemical abnormality. Unusual chemistry can accompany good wine; poor wine can have ordinary chemistry.

Define low quality as a rating at most 4 for this comparison. We introduce this criterion after calculating the anomaly scores.

In [ ]:
low_quality = red["quality"] <= 4
print("Low-rated records:", int(low_quality.sum()), "of", len(red))

selected = flags["LOF"]
tp = int((selected & low_quality).sum())
fp = int((selected & ~low_quality).sum())
fn = int((~selected & low_quality).sum())
tn = int((~selected & ~low_quality).sum())
confusion = pd.DataFrame(
    [[tp, fp], [fn, tn]],
    index=["Flagged", "Unflagged"],
    columns=["Quality ≤ 4", "Quality > 4"],
)
display(confusion)

**A confusion matrix counts agreement and disagreement with a specified outcome criterion.** Its rows show the review decision; its columns show the sensory outcome.

- **True positives:** seven flagged low-rated wines.
- **False positives:** 43 flagged wines with ratings above 4.
- **False negatives:** 56 unflagged low-rated wines.
- **True negatives:** 1,493 unflagged wines with ratings above 4.

These terms describe agreement with low ratings. A “false positive” can still have unusual chemistry worth investigating.

**Precision is the fraction of flagged records meeting the criterion. Recall is the fraction of all records meeting the criterion that were flagged.**

| Measure | Calculation | Question |
| --- | --- | --- |
| Precision | 7 / 50 = 14% | Among the reviewed wines, how many have low ratings? |
| Recall | 7 / 63 ≈ 11.1% | How much of the low-rated group did we find? |

The first denominator measures workload; the second measures the outcome group available to find.

In [ ]:
precision = tp / (tp + fp)
recall = tp / (tp + fn)
print(f"Precision: {tp} / {tp + fp} = {precision:.1%}")
print(f"Recall: {tp} / {tp + fn} = {recall:.1%}")

## Experiment With The Threshold

Before lowering the threshold, predict what must happen to review count and recall. Can you predict the direction of precision?

Open the companion **Thresholds And Review Decisions** marimo app. Drag the vertical boundary through the score dots. The shaded region shows the selected records; the two rows distinguish low ratings from other ratings.

1. Start with LOF at 50 reviews, then select 100. How many extra low-rated wines do the additional reviews find?
2. Select all records, then none. Explain the precision and recall using their denominators.
3. Compare methods at 50 reviews. Their numerical thresholds differ; their workloads are the same.

Release the boundary to update the detailed confusion matrix and record table. Return here at **Compare Review Outcomes**.

Launch the companion from this notebook’s folder (see the [quickstart](10a-threshold-explorer-quickstart.md)):

```bash
uvx --from 'marimo==0.25.1' marimo run --sandbox 10a-threshold-explorer.py
```

The Python source and `evidence-scores.csv` travel together. The tables below preserve the comparison when the interactive app is unavailable.

##### Answer

Lowering the threshold cannot remove previously selected records, so workload and recall cannot decrease. Precision depends on the mix entering the list.

LOF's extra 50 reviews add three low-rated wines and 47 other wines. Recall rises from 11.1% to 15.9%, while precision falls from 14% to 10%. Selecting all gives recall 100% and precision about 3.9%; selecting none gives recall zero and undefined precision.

## Compare Review Outcomes

Keep the outcome criterion fixed and compare the methods at equal review counts. The helper below applies each threshold and counts the selected low-rated wines. It is shared by the following comparisons.

In [ ]:
def review_outcome(score, count):
    cutoff = score.nlargest(count).iloc[-1]
    chosen = score >= cutoff
    found = int((chosen & low_quality).sum())
    return {
        "Reviewed": int(chosen.sum()),
        "Low-rated found": found,
        "Low-rated missed": int(low_quality.sum()) - found,
        "Precision": found / chosen.sum(),
        "Recall": found / low_quality.sum(),
    }

rows = []
for method in scores.columns:
    for count in [25, 50, 100]:
        rows.append({"Method": method, **review_outcome(scores[method], count)})
review_results = pd.DataFrame(rows)
display(review_results.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for method in scores.columns:
    ordered = scores[method].sort_values(ascending=False, kind="stable").index
    found = low_quality.loc[ordered].to_numpy()[:200].cumsum()
    ax.step(np.arange(1, 201), found, where="post", label=method)
ax.plot(np.arange(1, 201), np.arange(1, 201) * low_quality.mean(),
        ":", color="black", label="Random selection: expected count")
ax.set(xlabel="Records reviewed", ylabel="Low-rated wines found",
       title="Workload and agreement with low ratings")
ax.legend()
plt.show()

A flat stretch means additional reviews add other wines. Isolation Forest adds no low-rated wines between 25 and 50 reviews, then adds four between 50 and 100. LOF finds more low-rated wines at each displayed capacity.

At 50 reviews, LOF therefore shows stronger agreement with low ratings. Next, check whether a reasonable change in its neighborhood setting changes that result.

## Check Whether A Setting Changes The Decision

**Sensitivity analysis examines how a result changes when an analytical choice changes. Stability describes how consistently it persists across specified changes.** Here we ask whether an uncertain setting changes whom we investigate or our conclusion about usefulness.

Start with LOF's 20-neighbor, 50-record selection. Change only the neighborhood size to 10, refit, and select 50 again.

In [ ]:
lof10 = LocalOutlierFactor(n_neighbors=10).fit(X)
score10 = pd.Series(-lof10.negative_outlier_factor_, index=red.index)
chosen10 = score10 >= score10.nlargest(50).iloc[-1]
shared = int((chosen10 & flags["LOF"]).sum())
display(pd.DataFrame([
    {"Neighbors": 20, **review_outcome(scores["LOF"], 50)},
    {"Neighbors": 10, **review_outcome(score10, 50)},
]).round(3))
print("Shared selections:", shared, "of 50")

Seventeen proposed investigations change, and the count of low-rated wines falls from seven to four. The recommendation therefore depends partly on the neighborhood choice.

This gives us a concrete question to investigate: Which neighborhood size helps us find the kinds of unusual observations we want to investigate?

## Contamination And Masking

Two additional effects can make it difficult to identify anomalies.

**Reference contamination occurs when the fitted reference contains behavior we intend to detect as abnormal.** Those observations can influence the pattern used to assess them. Could a concerning observation receive a low score because of that influence?

**Masking occurs when abnormal observations make one another appear less unusual.** The following construction keeps the accepted background and one focal observation fixed while adding more abnormal neighbors.

Generate 200 accepted observations near the origin and a tight remote group. By construction, the remote group contains the observations of interest. Compare five remote observations with thirty, always using 20 LOF neighbors. The circle identifies the same observation in both panels.

In [ ]:
rng = np.random.default_rng(7130)
background = rng.normal(0, .45, (200, 2))
remote = rng.normal([4, 0], .09, (30, 2))
masking_rows = []
masking_scores = []
for size in [5, 30]:
    combined = np.vstack([background, remote[:size]])
    fitted = LocalOutlierFactor(n_neighbors=20).fit(combined)
    neighbors = fitted.kneighbors(n_neighbors=20, return_distance=False)
    focal_score = -fitted.negative_outlier_factor_[200]
    masking_rows.append({
        "Remote group size": size,
        "Focal neighbors from remote group": int((neighbors[200] >= 200).sum()),
        "Focal LOF": focal_score,
    })
    masking_scores.append(focal_score)
display(pd.DataFrame(masking_rows).round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), sharex=True, sharey=True)
for ax, size, score in zip(axes, [5, 30], masking_scores, strict=True):
    ax.scatter(*background.T, s=12, color="gray")
    ax.scatter(*remote[:size].T, marker="^", color="#b34b22")
    ax.scatter(*remote[0], s=150, facecolors="none", edgecolors="black", linewidths=2)
    ax.set(title=f"{size} remote observations; focal LOF {score:.2f}",
           xlabel="Constructed feature 1", xlim=(-1.6, 4.5), ylim=(-1.6, 1.6))
axes[0].set_ylabel("Constructed feature 2")
fig.tight_layout()
plt.show()

We fit LOF to the whole collection, including the abnormal orange observations. Including anomalies in the fitted data is expected when searching a mixed collection. Here, however, the growing orange group provides enough mutual support to make its LOF scores approach 1. This is masking: the abnormal observations influence their own reference enough to become harder to detect.

### Explain The Score Change

The focal observation has not moved. Why does its LOF fall from about 6.19 to 1.01? Use the number of nearby companions and the 20-neighbor requirement to explain the difference.

##### Answer

With five remote observations, the focal record has four nearby companions and needs 16 background observations to complete its neighborhood. It is weakly supported relative to those neighbors.

With thirty remote observations, all 20 neighbors can come from the remote group. They provide comparable local support, so the LOF approaches 1. Similar abnormal observations have changed the fitted reference enough to conceal the departure.

### Compare Isolation Forest

Does Isolation Forest respond the same way? Fit it to the same two collections and inspect the circled observation's rank, with rank 1 indicating the highest anomaly score. Keep 300 trees, the seed, and 200 sampled observations per tree fixed across the two fits.

In [ ]:
method_comparison = []
for size in [5, 30]:
    combined = np.vstack([background, remote[:size]])
    local_model = LocalOutlierFactor(n_neighbors=20).fit(combined)
    isolation_model = IsolationForest(
        n_estimators=300, max_samples=200, random_state=7130
    ).fit(combined)
    for method, values in [
        ("LOF", -local_model.negative_outlier_factor_),
        ("Isolation Forest", -isolation_model.score_samples(combined)),
    ]:
        focal_rank = int((values > values[200]).sum()) + 1
        method_comparison.append({
            "Remote group": size, "Method": method,
            "Focal rank": focal_rank, "Total records": len(combined),
        })
display(pd.DataFrame(method_comparison))

LOF's focal rank falls from 5 of 205 to 125 of 230. Isolation Forest's falls from 2 of 205 to 25 of 230. Both give the observation less priority as the remote group grows, but Isolation Forest keeps it much nearer the top of the list. The methods respond differently to this form of reference contamination.

### Investigate The Reference

The following table summarizes ways to investigate a questionable reference and decide how to respond.

| Concern | Possible Response |
| --- | --- |
| Accepted-normal observations are available. | Use them to establish the reference. |
| Legitimate populations follow different patterns. | Compare relevant populations separately, as with red and white wine. |
| Suspicious groups may reflect faults, errors, or repeated exports. | Investigate provenance and condition before excluding them. |
| The appropriate reference or scale is uncertain. | Compare plausible choices and inspect decision consequences. |

Domain evidence helps distinguish an abnormal group from a legitimate minority population. Excluding the group or increasing the neighborhood size changes the comparison, so inspect the consequences of either choice.

For a collective anomaly, the unexpected feature is the relationship or joint pattern among observations. In the masking example, we designated the orange group as abnormal and examined how its members affected one another's scores.

### Separate Reference Contamination From The Parameter

**The `contamination` parameter specifies an assumed abnormal fraction used to choose a score threshold.** In these implementations, changing it changes flags without changing raw scores. The same observations remain in the fitted reference.

The next cell checks this distinction for LOF on the same wine inputs.

In [ ]:
cutoff_rows = []
for fraction in [.02, .10]:
    fitted = LocalOutlierFactor(n_neighbors=20, contamination=fraction).fit(X)
    raw_scores = -fitted.negative_outlier_factor_
    assert np.array_equal(raw_scores, scores["LOF"].to_numpy())
    cutoff_rows.append({
        "contamination": fraction,
        "Flags": int((fitted.negative_outlier_factor_ < fitted.offset_).sum()),
        "Raw scores": "unchanged",
    })
display(pd.DataFrame(cutoff_rows))

The parameter changes the flag count from 32 to 160. It provides another way to set a score boundary, alongside a manual threshold or review-budget rule. The library uses its own strict comparison at that boundary.

The fitted density relationships stay unchanged. A masked group with ordinary-looking scores may remain below many other records even when we flag more. Address that problem by investigating the reference and method, alongside the threshold.

## Test Agreement Against Random Selection

**Validation evaluates whether evidence supports the intended claim.** LOF finds seven low-rated wines in 50 reviews. A uniform random selection would find about 50 × 63 / 1599 ≈ 1.97 on average. Use a one-sided hypothesis test to assess that difference.

- **Null hypothesis:** the selected set has no greater concentration of low ratings than a uniform random selection of 50 records.
- **Alternative hypothesis:** the selected set has a greater concentration of low ratings.
- **Test statistic:** the number of low-rated wines among the 50 selected records; the observed value is seven.

Construct the null distribution by repeatedly selecting 50 distinct records uniformly at random from this collection. Keep the observed ratings, collection size, and review count fixed.

In [ ]:
test_rng = np.random.default_rng(7130)
outcomes = low_quality.to_numpy()
random_counts = []
for _repetition in range(20000):
    selected_positions = test_rng.choice(len(red), size=50, replace=False)
    random_counts.append(int(outcomes[selected_positions].sum()))
random_counts = np.array(random_counts)
observed = int((flags["LOF"] & low_quality).sum())
print("Observed count:", observed)
tail_count = int((random_counts >= observed).sum())
print(f"Random selections finding at least this many: {tail_count} / {len(random_counts)}")
print("Estimated one-sided p-value:", (random_counts >= observed).mean())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
counts = np.arange(10)
frequencies = np.bincount(random_counts, minlength=10)[:10] / len(random_counts)
ax.bar(counts, frequencies, color=["#b34b22" if n >= observed else "#547e9e" for n in counts])
ax.axvline(observed, linestyle="--", color="black", label="Observed LOF count")
ax.set(xlabel="Low-rated wines in a random selection of 50", ylabel="Fraction of selections",
       title="Random-review comparison", xticks=counts)
ax.legend()
plt.show()

Only 53 of the 20,000 random selections find at least seven low-rated wines. The estimated one-sided p-value is 53 / 20000 = .00265.

The exact count distribution is hypergeometric because each selection samples without replacement. Its upper-tail probability gives the exact p-value: `sf(6, ...)` counts outcomes greater than six, hence at least seven.

In [ ]:
exact_probability = hypergeom.sf(observed - 1, len(red), int(low_quality.sum()), 50)
p_value = exact_probability
print(f"Exact one-sided p-value: {p_value:.6f}")

### Interpret The Evidence

The p-value is about .00275. Using a 5% significance level for this test, what conclusion follows about agreement between the selected records and low ratings?

##### Answer

The p-value is below .05, so reject the null hypothesis for this fixed selection. The selected records contain more low-rated wines than uniform random selection would commonly produce.

This is evidence of an association between the anomaly flags and the external quality criterion. Its practical value depends on the purpose of the investigation and the effort required to review the selected records.

### Evaluate A Chosen Procedure

Trying several settings and keeping the one with the best result can make performance look better than it will be on another collection. Once we choose a detector and threshold rule, evaluate that choice using records whose outcomes did not guide the choice.

For this exercise, the p-value describes the specified 50-record selection. Because we have inspected several methods and settings, a fresh evaluation is the next step before relying on the chosen procedure.

## Evaluate New Observations

**Evaluation on new observations checks how a chosen procedure performs beyond the data used to develop it.** Suppose the laboratory adopts the 20-neighbor LOF procedure and a capacity of 50 reviews per collection.

1. **Fix the procedure.** Keep the chemical features, fitted scaler, reference model, and review-budget rule chosen during development.
2. **Score a separate collection.** Transform its chemical measurements with the existing scaler and calculate scores against the fitted reference.
3. **Select 50 records.** Apply the budget rule before looking at their quality ratings. The numerical threshold comes from this new collection's ranking.
4. **Evaluate the outcomes.** Count low-rated wines among the selected records and compare the result with random selections of the same size from the new collection.

This tests whether the agreement seen during development carries over. With a fixed numerical threshold instead, allow the number selected to vary and report that workload too.

Keep repeated measurements or related samples together when separating development and evaluation data, so the evaluation represents new observations.

### Novelty Detection

**Novelty detection assesses new observations against an established reference representing normal behavior.** We will assume that the wine collection can stand as an established reference for this section. To carry out the new-observation evaluation above with LOF, fit the chosen reference using its novelty interface.

A fitted Isolation Forest already contains split rules. If a tree first splits at 25 and then splits its right branch at 65, a new value of 30 travels right, then left. Its path contributes to the score across the fitted forest.

**Scoring preserves the fitted reference; refitting changes it.** The masking example refitted after adding observations. With a fixed novelty reference, newly scored observations do not become reference neighbors merely by being scored.

| Task | Isolation Forest | LOF |
| --- | --- | --- |
| Score new observations | Apply `score_samples` to new inputs. | Fit with `novelty=True`, then apply `score_samples` to unseen inputs. |
| Prepare new inputs | Apply the scaler fitted on the reference. | Apply the scaler fitted on the reference. |
| Update the reference | Deliberately refit when justified. | Deliberately refit when justified. |

LOF's novelty calls should not be used to reproduce its training scores; those remain available through `negative_outlier_factor_`. Deciding when to accept new behavior into the reference is part of the application.

Outcome evidence determines what we can evaluate. If ratings are available only for the 50 reviewed wines, seven low ratings give precision of 14%. To calculate recall, we also need the total number of low-rated wines in the collection. Sampling unselected records for review helps assess what the policy misses.

## Make An Investigation Recommendation

At a capacity of 50, LOF's selected records contain more low-rated wines than Isolation Forest's selection and more than random selection commonly would. That association gives us a reason to investigate the relationship between chemical unusualness and sensory quality further.

Give a short recommendation that names what you would investigate next. Support it with one result and identify the evidence that would help you decide whether to use the procedure.

##### Answer

One reasonable recommendation is to review the LOF candidates and evaluate the chosen procedure on a separate collection. Seven of its 50 candidates have low ratings, and the statistical comparison shows a stronger association than random review commonly produces. Laboratory rechecks can then establish whether the unusual profiles identify problems worth the review effort.

The reference and method determine the scores; the threshold determines attention; investigation and external outcomes establish what the selections mean.